<a href="https://colab.research.google.com/github/Juliodelemos/sugarcane-variety-classification-yolov8/blob/main/training_10_seeds_ipyn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install ultralytics

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# Note: Google Drive mounting is not required for the public version if the dataset is downloaded locally or from a public repository.

In [ ]:
import os
import shutil
import random
import numpy as np
import pandas as pd
import scipy.stats as st
from ultralytics import YOLO

# ==========================================
# 1. DIRECTORY CONFIGURATION AND SEEDS
# ==========================================
# Insert here the path to the dataset downloaded from REDU.
DATASET_ORIGINAL = 'Dataset_Original/'
# Insert here the path to the directory where the results will be saved.
BASE_WORK_DIR = 'runs_statistics/'

# Test parameters
SEEDS = [42, 101, 2023, 777, 88, 1234, 555, 999, 333, 111]
EPOCHS = 30
PATIENCE = 10
SPLIT_RATIO = (0.7, 0.15, 0.15) # Train, Validation, Test

# Lists to store the results
final_results = []
best_accuracy = 0.0

print("Starting the rigorous testing battery...\n")

# ==========================================
# 2. 10-ROUNDS LOOP
# ==========================================
for i, seed in enumerate(SEEDS):
    run_name = f'run_seed_{seed}'
    run_dir = os.path.join(BASE_WORK_DIR, run_name)
    dataset_temp = os.path.join(run_dir, 'dataset_split')

    print(f"\n--- Starting Round {i+1}/10 (Seed: {seed}) ---")

    # 2.1 Create temporary folder structure
    for split in ['train', 'val', 'test']:
        for classe in os.listdir(DATASET_ORIGINAL):
            os.makedirs(os.path.join(dataset_temp, split, classe), exist_ok=True)

    # 2.2 Dynamic Data Splitting (Random Subsampling)
    random.seed(seed)
    for classe in os.listdir(DATASET_ORIGINAL):
        classe_path = os.path.join(DATASET_ORIGINAL, classe)
        if not os.path.isdir(classe_path): continue

        imagens = os.listdir(classe_path)
        random.shuffle(imagens)

        n_train = int(len(imagens) * SPLIT_RATIO[0])
        n_val = int(len(imagens) * SPLIT_RATIO[1])

        treino = imagens[:n_train]
        validacao = imagens[n_train:n_train+n_val]
        teste = imagens[n_train+n_val:]

        # Copy images to the temporary split
        for img in treino: shutil.copy(os.path.join(classe_path, img), os.path.join(dataset_temp, 'train', classe, img))
        for img in validacao: shutil.copy(os.path.join(classe_path, img), os.path.join(dataset_temp, 'val', classe, img))
        for img in teste: shutil.copy(os.path.join(classe_path, img), os.path.join(dataset_temp, 'test', classe, img))

    # 2.3 YOLOv8 Classify Training
    print(f"Training model with seed {seed}...")
    model = YOLO('yolov8n-cls.pt')

    # Train
    results = model.train(
        data=dataset_temp,
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=224,
        seed=seed,
        project=run_dir,
        name='train',
        exist_ok=True,
        verbose=False
    )

    # 2.4 Metrics Extraction (Top1 and Top5 Accuracy from results object)
    top1_acc = results.top1
    top5_acc = results.top5

    final_results.append({
        'Seed': seed,
        'Top1_Accuracy': top1_acc,
        'Top5_Accuracy': top5_acc
    })

    print(f"Round {i+1} completed -> Top-1 Accuracy: {top1_acc:.4f}")

    # 2.5 Space Management (Smart Cleanup)
    # Delete the split images to save space
    shutil.rmtree(dataset_temp)

    # If it is not the best model, delete the heavy weights (.pt) from the project folder
    peso_treinado = os.path.join(run_dir, 'train', 'weights', 'best.pt')
    if top1_acc > best_accuracy:
        best_accuracy = top1_acc
        print(">>> New best model found! Keeping saved weights.")
    else:
        # Delete the weights folder of this round to save space
        if os.path.exists(os.path.join(run_dir, 'train', 'weights')):
            shutil.rmtree(os.path.join(run_dir, 'train', 'weights'))

# ==========================================
# 3. STATISTICAL CALCULATION AND ANALYSIS (95% CI)
# ==========================================
df_results = pd.DataFrame(final_results)

# Function to calculate 95% Confidence Interval
def compute_ci(data, confidence=0.95):
    n = len(data)
    m = np.mean(data)
    se = st.sem(data)
    h = se * st.t.ppf((1 + confidence) / 2., n-1)
    return m, h

mean_acc, ci_acc = compute_ci(df_results['Top1_Accuracy'])

print("\n" + "="*50)
print(" FINAL STATISTICAL RESULTS FOR THE ARTICLE")
print("="*50)
print(df_results.to_string(index=False))
print("-" * 50)
print(f"Global Mean Accuracy  : {mean_acc:.4f}")
print(f"Standard Deviation    : {df_results['Top1_Accuracy'].std():.4f}")
print(f"Confidence Interval (95%): {mean_acc:.4f} ± {ci_acc:.4f}")
print(f"That is, we are 95% confident that the true accuracy is between: [{mean_acc-ci_acc:.4f}, {mean_acc+ci_acc:.4f}]")
print("="*50)

# Save final table to CSV for the article
df_results.to_csv(os.path.join(BASE_WORK_DIR, 'final_statistical_table.csv'), index=False)
